Additional evaluations on outputs, in addition to those in model training/cv

In [2]:
import pandas as pd
import re
import os
from tensorflow.keras.layers import Dot, Activation
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
# import tensorflow_recommenders as tfrs

# 👉 everything below comes from *tf.keras*
from tensorflow import keras
from tensorflow.keras.utils   import FeatureSpace
from tensorflow.keras.layers  import TextVectorization

from keras_rs.layers import BruteForceRetrieval
from keras_rs.metrics import PrecisionAtK, RecallAtK
from keras.losses import BinaryFocalCrossentropy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.metrics                 import roc_auc_score

import numpy as np, pandas as pd, tensorflow as tf, keras
# from keras.layers import TextVectorization, Embedding, Concatenate
# from keras.utils  import FeatureSpace
from sklearn.metrics import classification_report

2025-12-01 12:31:42.316101: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-01 12:31:42.326798: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764585102.338704    2588 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764585102.343068    2588 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764585102.353117    2588 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
DATA_DIR = "../data/opentargets/"

In [4]:
# df_learn = pd.read_parquet("../data/proc/df_learn.parquet")
# print(df_learn.shape)
# display(df_learn)
disease_df = pd.read_parquet("../data/proc/disease_df.parquet")
print(disease_df.shape)
display(disease_df.head(2))
target_df = pd.read_parquet("../data/proc/target_df.parquet")
print(target_df.shape)
display(target_df.head(2))

(38959, 9)


,diseaseId,name,description,dbXRefs,synonyms,ancestors,therapeuticAreas,ExactSynonyms,disease_text_embed
0,DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...,"[MESH:D000080874, MONDO:0000510, UMLS:C5191670...",NaN,"[MONDO_0024237, EFO_0005772, EFO_0000618, MOND...","[EFO_0000618, OTAR_0000018, OTAR_0000020]",alpha Synucleinopathies synucleinopathy,synucleinopathy alpha Synucleinopathies synucl...
1,DOID_10113,trypanosomiasis,Infection with protozoa of the genus trypanosoma.,"[UMLS:C0041227, MONDO:0000940, ICD10CM:B56, Me...",NaN,"[MONDO_0002428, EFO_0001067, EFO_0005741]",[EFO_0005741],Trypanosoma caused disease or disorder Trypano...,trypanosomiasis Trypanosoma caused disease or ...


(17065, 35)


,targetId,approvedSymbol,biotype,genomicLocation,alternativeGenes,approvedName,go,hallmarks,synonyms,functionDescriptions,...,count_tractability,count_alternativeGenes,count_hallmarks,count_functionDescriptions,count_tep,sym,target_text_embed,score_syn,score_mis,score_lof
0,ENSG00000000457,SCYL3,protein_coding,"{'chromosome': '1', 'end': 169894267, 'start':...",None,SCY1 like pseudokinase 3,"[GO:0005737, GO:0042802, GO:0016477, GO:000551...",NaN,,May play a role in regulating cell adhesion/mi...,...,1,0,0,1,0,SCYL,SCYL SCY1 like pseudokinase 3 May play a role...,0.70818,0.98492,0.28151
1,ENSG00000001167,NFYA,protein_coding,"{'chromosome': '6', 'end': 41102403, 'start': ...",None,nuclear transcription factor Y subunit alpha,"[GO:0000785, GO:0005515, GO:0016602, GO:000635...",NaN,,Component of the sequence-specific heterotrime...,...,2,0,0,1,0,NFYA,NFYA nuclear transcription factor Y subunit al...,-0.17463,2.78030,0.14619


In [5]:
# Ensure columns are treated as strings to avoid errors with NaN values
disease_df['diseaseId'] = disease_df['diseaseId'].astype(str)
disease_df['dbXRefs'] = disease_df['dbXRefs'].astype(str)

# --- ONE-LINERS START ---

# 1. Define Rare Disease (Exact match for Orphanet ID or reference)
disease_df['is_rare'] = disease_df['diseaseId'].str.contains('Orphanet|ORPHA') | disease_df['dbXRefs'].str.contains('Orphanet|ORPHA')
print(disease_df['is_rare'].value_counts())
# 2. Define Mendelian/Simple Disease (Exact match for OMIM ID or reference)
disease_df['has_omim_annotation'] = disease_df['diseaseId'].str.contains('OMIM') | disease_df['dbXRefs'].str.contains('OMIM')
print(disease_df['has_omim_annotation'].value_counts())


is_rare
False    29803
True      9156
Name: count, dtype: int64
has_omim_annotation
False    31460
True      7499
Name: count, dtype: int64


In [8]:
df_preds = pd.read_csv("DL_novel_candidates_predictions.csv") # DL preds, novel and positive
df_preds

,diseaseId,diseaseName,targetId,targetSymbol,score,label,source,disease_num_known_clinical_targets,orphan
0,UBERON_0000104,life cycle,ENSG00000123201,GUCY1B2,0.948,-1,model_prediction,0,True
1,UBERON_0000104,life cycle,ENSG00000261456,TUBB8,0.824,-1,model_prediction,0,True
2,UBERON_0000104,life cycle,ENSG00000088832,FKBP1A,0.734,-1,model_prediction,0,True
3,UBERON_0000104,life cycle,ENSG00000176014,TUBB6,0.710,-1,model_prediction,0,True
4,UBERON_0000104,life cycle,ENSG00000173213,TUBB8B,0.700,-1,model_prediction,0,True
...,...,...,...,...,...,...,...,...,...
147228,DOID_10113,trypanosomiasis,ENSG00000151834,GABRA2,1.000,1,known_positive,11,False
147229,DOID_10113,trypanosomiasis,ENSG00000163285,GABRG1,1.000,1,known_positive,11,False
147230,DOID_10113,trypanosomiasis,ENSG00000182256,GABRG3,1.000,1,known_positive,11,False
147231,DOID_10113,trypanosomiasis,ENSG00000183454,GRIN2A,1.000,1,known_positive,11,False


In [9]:
df_novel = df_preds.query("label<1")
df_novel

,diseaseId,diseaseName,targetId,targetSymbol,score,label,source,disease_num_known_clinical_targets,orphan
0,UBERON_0000104,life cycle,ENSG00000123201,GUCY1B2,0.948,-1,model_prediction,0,True
1,UBERON_0000104,life cycle,ENSG00000261456,TUBB8,0.824,-1,model_prediction,0,True
2,UBERON_0000104,life cycle,ENSG00000088832,FKBP1A,0.734,-1,model_prediction,0,True
3,UBERON_0000104,life cycle,ENSG00000176014,TUBB6,0.710,-1,model_prediction,0,True
4,UBERON_0000104,life cycle,ENSG00000173213,TUBB8B,0.700,-1,model_prediction,0,True
...,...,...,...,...,...,...,...,...,...
147217,DOID_10113,trypanosomiasis,ENSG00000268089,GABRQ,0.874,-1,model_prediction,11,False
147218,DOID_10113,trypanosomiasis,ENSG00000069696,DRD4,0.871,-1,model_prediction,11,False
147219,DOID_10113,trypanosomiasis,ENSG00000145863,GABRA6,0.869,-1,model_prediction,11,False
147220,DOID_10113,trypanosomiasis,ENSG00000151577,DRD3,0.868,-1,model_prediction,11,False


In [10]:
df_known_pos = df_preds.query("label>0")
df_known_pos

,diseaseId,diseaseName,targetId,targetSymbol,score,label,source,disease_num_known_clinical_targets,orphan
22,Orphanet_99842,Leukocyte adhesion deficiency type I,ENSG00000113302,IL12B,1.0,1,known_positive,1,False
33,Orphanet_98974,Fuchs endothelial corneal dystrophy,ENSG00000067900,ROCK1,1.0,1,known_positive,2,False
34,Orphanet_98974,Fuchs endothelial corneal dystrophy,ENSG00000134318,ROCK2,1.0,1,known_positive,2,False
67,Orphanet_98920,Spinal muscular atrophy with respiratory distr...,ENSG00000172062,SMN1,1.0,1,known_positive,2,False
68,Orphanet_98920,Spinal muscular atrophy with respiratory distr...,ENSG00000205571,SMN2,1.0,1,known_positive,2,False
...,...,...,...,...,...,...,...,...,...
147228,DOID_10113,trypanosomiasis,ENSG00000151834,GABRA2,1.0,1,known_positive,11,False
147229,DOID_10113,trypanosomiasis,ENSG00000163285,GABRG1,1.0,1,known_positive,11,False
147230,DOID_10113,trypanosomiasis,ENSG00000182256,GABRG3,1.0,1,known_positive,11,False
147231,DOID_10113,trypanosomiasis,ENSG00000183454,GRIN2A,1.0,1,known_positive,11,False



Load targetability - and max clinical phase reached
- max clinical trial pahse is per target, NOT target X disease

In [11]:
target_priority = pd.read_parquet(os.path.join(DATA_DIR, "target_prioritisation")).dropna(subset=["maxClinicalTrialPhase"])
target_priority

,targetId,isInMembrane,isSecreted,hasSafetyEvent,hasPocket,hasLigand,hasSmallMoleculeBinder,geneticConstraint,paralogMaxIdentityPercentage,mouseOrthologMaxIdentityPercentage,isCancerDriverGene,hasTEP,mouseKOScore,hasHighQualityChemicalProbes,maxClinicalTrialPhase,tissueSpecificity,tissueDistribution
6,ENSG00000004779,0.0,0.0,NaN,0.0,0.0,0.0,-0.197333,0.000000,0.294870,NaN,NaN,NaN,NaN,1.00,0.50,-1.0
11,ENSG00000005961,1.0,0.0,NaN,1.0,1.0,1.0,-0.188060,0.000000,0.000000,NaN,NaN,-0.859673,NaN,1.00,0.50,0.0
13,ENSG00000006327,1.0,0.0,NaN,0.0,0.0,0.0,0.636799,NaN,0.069765,NaN,NaN,-0.212319,NaN,0.50,-1.00,-1.0
21,ENSG00000007402,1.0,0.0,NaN,0.0,1.0,1.0,-0.850698,0.000000,0.842520,NaN,NaN,-0.947966,NaN,1.00,0.50,0.0
29,ENSG00000008988,0.0,0.0,NaN,0.0,1.0,1.0,0.126276,NaN,1.000000,NaN,NaN,-0.781258,NaN,1.00,-1.00,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74487,ENSG00000243646,0.0,0.0,NaN,0.0,0.0,0.0,0.250573,0.000000,0.000000,NaN,NaN,-0.103605,NaN,0.75,-1.00,-1.0
74556,ENSG00000248144,1.0,0.0,-1.0,1.0,0.0,1.0,NaN,-0.860000,0.253335,NaN,NaN,-0.520853,NaN,1.00,0.75,0.0
75476,ENSG00000262406,0.0,1.0,NaN,1.0,1.0,1.0,NaN,0.000000,0.000000,NaN,NaN,-0.089497,1.0,0.75,0.50,0.5
75768,ENSG00000268651,0.0,0.0,NaN,NaN,NaN,NaN,NaN,-1.000000,NaN,NaN,NaN,-0.117190,NaN,0.50,1.00,1.0


In [15]:
temp = df_known_pos.merge(target_priority,on=["targetId"],how="inner")
print(temp.shape[0])
print(temp.maxClinicalTrialPhase.corr(temp.score))

67532
nan


/home/ddofer/anaconda3/envs/ot-recsys-kerasrs/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ddofer/anaconda3/envs/ot-recsys-kerasrs/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
